# Validation Error Analysis

Review validation-only diagnostics for the strongest non-dummy model families from a completed ablation run. This notebook intentionally uses `features_val.csv` and ablation validation predictions only; do not point it at the held-out test table.

The setup cell below imports the plotting and analysis helpers, then defines the run directory and input paths used throughout the notebook.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

from src.models.error_analysis import (
    ValidationErrorAnalysisOutputs,
    run_validation_error_analysis,
)

RUN_DIR = Path("outputs/runs/full_ablation_20260718")
VALIDATION_FEATURES = Path("data/processed/features_val.csv")
FEATURE_MANIFEST = Path("data/processed/feature_manifest.csv")
EPOCH_INDEX = Path("data/interim/epoch_index.csv")
CLASS_ORDER = ["Wake", "Non-REM", "REM"]

## Load Or Refresh Analysis Artifacts

Keep `REFRESH_ANALYSIS_ARTIFACTS = False` to inspect artifacts that were already generated by `scripts/run_validation_error_analysis.py`. Set it to `True` only when you want the notebook to rebuild the validation error-analysis outputs. If refreshing, set `allow_partial=True` only for a smoke pass while an ablation run is still in progress; for final validation analysis, keep it `False` so missing ablation/model outputs fail loudly.

In [ ]:
REFRESH_ANALYSIS_ARTIFACTS = False
ANALYSIS_DIR = RUN_DIR / "error_analysis"


def load_cached_validation_error_analysis_outputs(
    analysis_dir: Path,
) -> ValidationErrorAnalysisOutputs:
    metrics_dir = analysis_dir / "metrics"
    figures_dir = analysis_dir / "figures"
    artifact_index_path = analysis_dir / "artifact_index.csv"
    permutation_importance_path = metrics_dir / "permutation_importance.csv"
    permutation_importance_path = (
        permutation_importance_path if permutation_importance_path.exists() else None
    )

    required_paths = [
        analysis_dir / "error_analysis_config.json",
        artifact_index_path,
        metrics_dir / "selected_models.csv",
        metrics_dir / "validation_predictions_with_context.csv",
        metrics_dir / "per_model_per_class_metrics.csv",
        metrics_dir / "per_participant_metrics.csv",
        metrics_dir / "transition_distance_metrics.csv",
        metrics_dir / "model_family_comparison.csv",
        metrics_dir / "feature_importance.csv",
        figures_dir / "model_family_macro_f1.png",
    ]
    missing = [path for path in required_paths if not path.exists()]
    if missing:
        missing_text = "\n".join(f"- {path}" for path in missing)
        raise FileNotFoundError(
            "Cached validation error-analysis artifacts are incomplete. "
            "Set REFRESH_ANALYSIS_ARTIFACTS = True to rebuild them.\n"
            f"Missing:\n{missing_text}"
        )

    artifact_index = pd.read_csv(artifact_index_path)
    return ValidationErrorAnalysisOutputs(
        analysis_dir=analysis_dir,
        metrics_dir=metrics_dir,
        figures_dir=figures_dir,
        config_path=analysis_dir / "error_analysis_config.json",
        artifact_index_path=artifact_index_path,
        selected_models_path=metrics_dir / "selected_models.csv",
        predictions_path=metrics_dir / "validation_predictions_with_context.csv",
        per_class_metrics_path=metrics_dir / "per_model_per_class_metrics.csv",
        per_participant_metrics_path=metrics_dir / "per_participant_metrics.csv",
        transition_metrics_path=metrics_dir / "transition_distance_metrics.csv",
        model_family_comparison_path=metrics_dir / "model_family_comparison.csv",
        feature_importance_path=metrics_dir / "feature_importance.csv",
        permutation_importance_path=permutation_importance_path,
        artifact_index=artifact_index,
    )


if REFRESH_ANALYSIS_ARTIFACTS:
    outputs = run_validation_error_analysis(
        run_dir=RUN_DIR,
        validation_features_path=VALIDATION_FEATURES,
        manifest_path=FEATURE_MANIFEST,
        epoch_index_path=EPOCH_INDEX,
        allow_partial=False,
        compute_permutation=True,
        permutation_repeats=5,
        compute_shap=True,
        max_shap_rows=1000,
        create_plots=True,
    )
else:
    outputs = load_cached_validation_error_analysis_outputs(ANALYSIS_DIR)

outputs.analysis_dir

## Selected Models

Load the completed ablation/model pairs selected for this analysis. This table is the review roster: it shows which feature set and model family each later diagnostic comes from, plus the validation and cross-validation summary scores used for quick orientation.

In [ ]:
selected_models = pd.read_csv(outputs.selected_models_path)
selected_models[[
    "ablation",
    "model",
    "n_features",
    "best_cv_macro_f1",
    "validation_macro_f1",
    "validation_accuracy",
]]

## Cross-Ablation Summary

Compare validation macro F1 across ablations and model families. Use this view to identify which feature groups materially improved or degraded validation performance before inspecting the more detailed error diagnostics.

In [ ]:
display(Image(filename=str(outputs.figures_dir / "model_family_macro_f1.png")))
pd.read_csv(outputs.model_family_comparison_path)

## Per-Class Metrics

Summarize class-specific performance as a compact pivot and heatmap. The default view uses per-class F1 so Wake, Non-REM, and REM errors can be compared across all selected ablation/model pairs without scrolling through the raw long-form table.

In [ ]:
per_class = pd.read_csv(outputs.per_class_metrics_path)
per_class_metric = "f1"
model_order = (
    selected_models["ablation"] + " / " + selected_models["model"]
).tolist()

per_class_pivot = (
    per_class.assign(
        model_label=lambda frame: frame["ablation"] + " / " + frame["model"]
    )
    .pivot_table(index="model_label", columns="label", values=per_class_metric)
    .reindex(index=model_order, columns=CLASS_ORDER)
)

display(
    per_class_pivot.style.format("{:.3f}").background_gradient(
        cmap="YlGnBu", vmin=0, vmax=1
    )
)

fig_height = max(4, 0.35 * len(per_class_pivot))
fig, ax = plt.subplots(figsize=(7, fig_height))
sns.heatmap(
    per_class_pivot,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
    cbar_kws={"label": per_class_metric.upper()},
    ax=ax,
)
ax.set_xlabel("True label")
ax.set_ylabel("Ablation / model")
ax.set_title(f"Per-class {per_class_metric.upper()}")
fig.tight_layout()
plt.show()

## Diagnostic Figures By Ablation And Model

Review the main diagnostics for each selected ablation/model pair. The notebook renders one normalized confusion matrix directly from the cached validation predictions, then displays the remaining curated figures while skipping older cached count confusion and native feature-importance images.

In [ ]:
artifact_index = pd.read_csv(outputs.artifact_index_path)
figure_index = artifact_index[artifact_index["artifact_type"] == "figure"].copy()
predictions = pd.read_csv(outputs.predictions_path)
skipped_figure_suffixes = (
    "_confusion_counts.png",
    "_confusion_true_normalized.png",
    "_feature_importance.png",
)


def display_normalized_confusion(predictions, ablation, model):
    subset = predictions[
        (predictions["ablation"] == ablation) & (predictions["model"] == model)
    ]
    confusion = pd.crosstab(subset["true_label"], subset["pred_label"])
    confusion = confusion.reindex(index=CLASS_ORDER, columns=CLASS_ORDER, fill_value=0)
    row_totals = confusion.sum(axis=1).replace(0, float("nan"))
    normalized = confusion.div(row_totals, axis=0).fillna(0.0)

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        normalized,
        annot=True,
        fmt=".2f",
        cmap="YlGnBu",
        cbar=False,
        vmin=0,
        vmax=1,
        ax=ax,
    )
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    fig.tight_layout()
    plt.show()

for row in selected_models.itertuples(index=False):
    print(f"\n=== {row.ablation} / {row.model} ===")
    display_normalized_confusion(predictions, row.ablation, row.model)
    subset = figure_index[
        (figure_index["ablation"] == row.ablation)
        & (figure_index["model"] == row.model)
    ]
    for figure_path in subset["path"]:
        if Path(figure_path).name.endswith(skipped_figure_suffixes):
            continue
        print(Path(figure_path).name)
        display(Image(filename=figure_path))

## Participant And Transition Tables

Inspect where performance is weakest by participant and by distance to the nearest true sleep-stage transition. The participant table surfaces the lowest macro-F1 cases first; the transition table helps separate boundary-adjacent errors from more stable-stage errors.

In [ ]:
participant_metrics = pd.read_csv(outputs.per_participant_metrics_path)
transition_metrics = pd.read_csv(outputs.transition_metrics_path)

display(participant_metrics.sort_values("macro_f1").head(20))
display(transition_metrics)

## Feature Importance Tables

Load the feature-importance CSV artifacts for targeted follow-up. Native model importance is retained as a table for reference, while the per-model figure review focuses on validation permutation importance because it is measured directly against validation macro F1.

In [ ]:
feature_importance = pd.read_csv(outputs.feature_importance_path)
display(feature_importance.head(50))

if outputs.permutation_importance_path is not None:
    permutation_importance = pd.read_csv(outputs.permutation_importance_path)
    display(permutation_importance.head(50))